# DailyIQ Keyframe Animator
Cartoon keyframes → RIFE in-between frames → 1080×1920 YouTube Short.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.
Then run the cells top to bottom (tap ▶ on each).

**Tips for good results**
- Keep poses close together (blink, head tilt, arm lift). RIFE smooths small moves well; big jumps blur or ghost.
- Same character, same size, same background in every keyframe.
- Name files `01_...`, `02_...` so they play in order (or use ORDER in cell 4).

In [ ]:
#@title 1) Setup — run once (about 1 minute)
import os, sys, subprocess, shutil, zipfile, urllib.request

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise RuntimeError(f"Command failed: {cmd}")
    return r.stdout

RIFE_MODEL = "4.25"  #@param ["4.25", "4.26"]
REPO = "/content/Practical-RIFE"
TL = f"{REPO}/train_log"

if not os.path.isdir(REPO):
    sh(f"git clone --depth 1 https://github.com/hzwer/Practical-RIFE {REPO}")

if not (os.path.exists(f"{TL}/flownet.pkl") and os.path.exists(f"{TL}/RIFE_HDv3.py")):
    url = f"https://huggingface.co/Bash2X/RIFE-Models/resolve/main/RIFE_v{RIFE_MODEL}.zip"
    zpath = "/content/rife_model.zip"
    print("Downloading RIFE weights...")
    urllib.request.urlretrieve(url, zpath)
    tmp = "/content/rife_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(tmp)
    src = None
    for root, _, fs in os.walk(tmp):
        if "flownet.pkl" in fs:
            src = root
            break
    if src is None:
        raise RuntimeError("flownet.pkl not found inside the model zip")
    os.makedirs(TL, exist_ok=True)
    for f in os.listdir(src):
        fp = os.path.join(src, f)
        if os.path.isfile(fp):
            shutil.copy(fp, TL)

if not os.path.exists(f"{TL}/RIFE_HDv3.py"):
    raise RuntimeError("RIFE_HDv3.py missing in train_log. Try the other model version in the dropdown.")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (set Runtime > T4 GPU, it will be very slow otherwise)")
print(sh("ffmpeg -version | head -1").strip())
print("Setup done ✓")

In [ ]:
#@title 2) Load RIFE
import os, sys, numpy as np, cv2, torch
import torch.nn.functional as F

sys.path.insert(0, REPO)
os.chdir(REPO)
from train_log.RIFE_HDv3 import Model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)
model = Model()
if not hasattr(model, "version"):
    model.version = 0
model.load_model(TL, -1)
model.eval()
model.device()

def _to_t(img):
    return torch.from_numpy(img.transpose(2, 0, 1).copy()).to(device).unsqueeze(0).float() / 255.0

def rife_mid(a, b, t):
    """a, b: HxWx3 uint8 RGB. t in (0,1). Returns the frame between them."""
    h, w = a.shape[:2]
    ph = ((h - 1) // 128 + 1) * 128
    pw = ((w - 1) // 128 + 1) * 128
    pad = (0, pw - w, 0, ph - h)
    I0 = F.pad(_to_t(a), pad)
    I1 = F.pad(_to_t(b), pad)
    try:
        out = model.inference(I0, I1, t, 1.0)
    except TypeError:
        out = model.inference(I0, I1, t)
    out = out[0].clamp(0, 1)[:, :h, :w]
    return (out.cpu().numpy().transpose(1, 2, 0) * 255).round().astype(np.uint8)

_test = rife_mid(np.zeros((256, 256, 3), np.uint8), np.full((256, 256, 3), 255, np.uint8), 0.5)
print("RIFE ready ✓  (test frame mean:", round(float(_test.mean()), 1), ")")

In [ ]:
#@title 3) Upload your keyframes (PNG/JPG/WEBP, or one .zip of them)
import os, shutil
from google.colab import files

KEY_DIR = "/content/keyframes"
shutil.rmtree(KEY_DIR, ignore_errors=True)
os.makedirs(KEY_DIR)
up = files.upload()
import zipfile
for name, data in up.items():
    p = os.path.join(KEY_DIR, name)
    with open(p, "wb") as f:
        f.write(data)
    if name.lower().endswith(".zip"):          # a zip of frames works too
        with zipfile.ZipFile(p) as z:
            for m in z.namelist():
                if m.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp")) and not m.startswith("__MACOSX"):
                    with open(os.path.join(KEY_DIR, os.path.basename(m)), "wb") as f:
                        f.write(z.read(m))
        os.remove(p)
print(len(os.listdir(KEY_DIR)), "keyframes ready")

In [ ]:
#@title 4) Settings
FPS = 30  #@param [24, 30]
SECONDS_PER_MOVE = 1.0  #@param {type:"number"}
HOLD_SECONDS = 0.3  #@param {type:"number"}
END_HOLD_SECONDS = 0.8  #@param {type:"number"}
SMOOTH_EASING = True  #@param {type:"boolean"}
FIT = "cover"  #@param ["cover", "contain"]
BACKGROUND = "#FFFFFF"  #@param {type:"string"}
ORDER = ""  #@param {type:"string"}
MOVES = ""  #@param {type:"string"}
# MOVES: how to get from each frame to the next, comma separated: smooth (RIFE), fade (quick crossfade), cut (instant).
# Example for 7 frames: smooth,smooth,smooth,cut,smooth,cut   (leave empty = smooth everywhere)
FADE_SECONDS = 0.15  #@param {type:"number"}
# ORDER: leave empty to play in filename order, or type e.g. 3,1,2,4 (numbers shown in cell 5's list)
AUDIO_PATH = None

In [ ]:
#@title 5) Make the video (silent)
import os, re, glob, time, subprocess
import numpy as np, cv2

W, H = 1080, 1920
OUT = "/content/dailyiq_short.mp4"

def natural(s):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", s)]

exts = (".png", ".jpg", ".jpeg", ".webp", ".bmp")
paths = sorted([p for p in glob.glob(KEY_DIR + "/*") if p.lower().endswith(exts)],
               key=lambda p: natural(os.path.basename(p)))
if ORDER.strip():
    idx = [int(x) - 1 for x in re.split(r"[,\s]+", ORDER.strip()) if x]
    paths = [paths[i] for i in idx]
if len(paths) < 2:
    raise ValueError("Upload at least 2 keyframes (cell 3).")
for i, p in enumerate(paths, 1):
    print(i, os.path.basename(p))

hx = BACKGROUND.lstrip("#")
bg = tuple(int(hx[i:i + 2], 16) for i in (0, 2, 4))  # RGB

def load_fit(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f"Cannot read {path}")
    if img.dtype == np.uint16:
        img = (img / 257).astype(np.uint8)
    elif img.dtype != np.uint8:
        img = img.astype(np.uint8)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    if img.shape[2] == 4:
        a = img[:, :, 3:4].astype(np.float32) / 255.0
        rgb = img[:, :, :3].astype(np.float32)
        bgr_bg = np.array(bg[::-1], np.float32)
        img = (rgb * a + bgr_bg * (1 - a)).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    s = max(W / w, H / h) if FIT == "cover" else min(W / w, H / h)
    nw, nh = max(1, round(w * s)), max(1, round(h * s))
    img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LANCZOS4)
    if FIT == "cover":
        x0, y0 = (nw - W) // 2, (nh - H) // 2
        return np.ascontiguousarray(img[y0:y0 + H, x0:x0 + W])
    canvas = np.empty((H, W, 3), np.uint8)
    canvas[:] = bg
    x0, y0 = (W - nw) // 2, (H - nh) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = img
    return canvas

def ease(t):
    return t * t * (3 - 2 * t) if SMOOTH_EASING else t

n_move = max(2, round(SECONDS_PER_MOVE * FPS))
hold_n = max(1, round(HOLD_SECONDS * FPS))
end_n = max(1, round(END_HOLD_SECONDS * FPS))

styles = [x.strip().lower() for x in MOVES.split(",") if x.strip()]
styles += ["smooth"] * (len(paths) - 1 - len(styles))
for x in styles[:len(paths) - 1]:
    if x not in ("smooth", "fade", "cut"):
        raise ValueError(f"Unknown move style '{x}' (use smooth, fade or cut)")
fade_n = max(1, round(FADE_SECONDS * FPS))
keys = [load_fit(p) for p in paths]
log = open("/content/ffmpeg.log", "w")
enc = subprocess.Popen(
    ["ffmpeg", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}", "-r", str(FPS), "-i", "-",
     "-c:v", "libx264", "-preset", "medium", "-crf", "18", "-pix_fmt", "yuv420p", "-movflags", "+faststart", OUT],
    stdin=subprocess.PIPE, stdout=log, stderr=log)

frames = 0
t0 = time.time()
for k in range(len(keys)):
    for _ in range(hold_n + (end_n if k == len(keys) - 1 else 0)):
        enc.stdin.write(keys[k].tobytes())
        frames += 1
    if k < len(keys) - 1:
        st = styles[k]
        if st == "smooth":
            for i in range(1, n_move):
                enc.stdin.write(rife_mid(keys[k], keys[k + 1], ease(i / n_move)).tobytes())
                frames += 1
        elif st == "fade":
            for i in range(1, fade_n + 1):
                t = i / (fade_n + 1)
                mix = ((1 - t) * keys[k].astype(np.float32) + t * keys[k + 1].astype(np.float32)).astype(np.uint8)
                enc.stdin.write(mix.tobytes())
                frames += 1
        print(f"move {k + 1}→{k + 2} ({st}) done  ({time.time() - t0:.0f}s)")
enc.stdin.close()
enc.wait()
log.close()
if enc.returncode != 0:
    print(open("/content/ffmpeg.log").read()[-1500:])
    raise RuntimeError("ffmpeg failed")
print(f"Silent video ready: {frames} frames, {frames / FPS:.1f}s, {W}x{H} @ {FPS}fps")

In [ ]:
#@title 6) (Optional) Upload voice-over — skip this cell if you have none
from google.colab import files
up = files.upload()
if up:
    AUDIO_PATH = "/content/" + list(up.keys())[0]
    with open(AUDIO_PATH, "wb") as f:
        f.write(list(up.values())[0])
    print("Voice-over:", AUDIO_PATH)

In [ ]:
#@title 7) Finish + download
import os, shutil, subprocess

FINAL = "/content/dailyiq_short_final.mp4"

def dur(p):
    return float(subprocess.check_output(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=nw=1:nk=1", p]).decode().strip())

if AUDIO_PATH and os.path.exists(AUDIO_PATH):
    vd, ad = dur(OUT), dur(AUDIO_PATH)
    pad = max(0.0, ad - vd)
    cmd = ["ffmpeg", "-y", "-i", OUT, "-i", AUDIO_PATH, "-map", "0:v", "-map", "1:a"]
    if pad > 0.01:
        # voice-over longer than the animation: hold the last frame
        cmd += ["-vf", f"tpad=stop_mode=clone:stop_duration={pad:.3f}",
                "-c:v", "libx264", "-crf", "18", "-pix_fmt", "yuv420p"]
    else:
        cmd += ["-c:v", "copy"]
    cmd += ["-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", FINAL]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise RuntimeError("mux failed")
else:
    shutil.copy(OUT, FINAL)

print(f"Final: {dur(FINAL):.1f}s, {os.path.getsize(FINAL) / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(FINAL)
except ImportError:
    print("Saved at", FINAL)